# BirdCLEF+ 2025 - Pretrained models, A3: probe retry on cached Bird-MAE-Base embeddings

Version 2 of this notebook only tests the linear head bias, on the cached 768-d mean-pooled embeddings from `sc4000-birdclef-m3-a3-birdmae-embed`. With standardised features and 30 epochs, we compare the default bias at max_lr 1e-3 (control) with the final-layer bias set to the training log prior at max_lr 1e-3 and 3e-3. The split, labels, window weights, standardisation and validation metrics are the same as in that notebook.

CPU only, no submission, no inference. Output: `probe_retry_results.json`, rewritten after every config.

In [ ]:
import ast, glob, json, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

T0 = time.time()
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
DEVICE = "cpu"       # CPU-only notebook, by design
print("device:", DEVICE, "| torch.cuda.is_available():", torch.cuda.is_available(),
      "| torch threads:", torch.get_num_threads(), "| os.cpu_count():", os.cpu_count())

# Same values as the A3 embed notebook's CFG (split, labels, training recipe)
CFG = dict(val_frac=0.2, seed=42, secondary_weight=0.5, batch_size=1024, weight_decay=1e-2)
EPOCHS = dict(linear=2 if SMOKE_TEST else 30, mlp=2 if SMOKE_TEST else 12)   # linear 30 = the original A3 probe
BUDGET_S = float(os.environ.get("M3_BUDGET_S", "1200"))   # a config is skipped if it is projected to end later than this
EMB_DIM = 768
BASELINE_FIRST, BASELINE_CLIP_MAX = 0.7495, 0.716         # original A3 linear probe, Kaggle T4 run
PRIOR_EPS = 1e-6                                          # prior clipped to [eps, 1-eps]; only classes with no train mass reach it

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{marker} not found under /kaggle/input")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
CACHE = find_dir("birdmae_embeddings.npy", "M3_EMBED_DIR")
print("Data:", DATA, "| M1 index:", MELS, "| embeddings:", CACHE)

## Split, labels and cached embeddings

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

# Copied from M1 train-v2 via the A3 notebook: same call, same array of clip numbers, same stratify, same seed.
idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va))

# The cache must belong to exactly this split and clip order
val_clips = pd.read_csv(CACHE / "val_clips.csv")
if not np.array_equal(val_clips["clip"].values, idx_va):
    raise RuntimeError("val_clips.csv does not match the recomputed split")
if "filename" in index.columns:
    if not (index["filename"].astype(str).values[idx_va] == val_clips["filename"].astype(str).values).all():
        raise RuntimeError("val_clips.csv filenames do not match train_index.csv")

E = np.load(CACHE / "birdmae_embeddings.npy")
windows = pd.read_csv(CACHE / "birdmae_windows.csv")
assert E.shape[1] == EMB_DIM and len(E) == len(windows), (E.shape, len(windows))
E = E.astype(np.float32)
print(f"Embeddings {E.shape} | {windows['clip'].nunique()} clips | loaded at {time.time() - T0:.0f}s", flush=True)

## Window weights, standardisation and validation sets (as in the A3 notebook)

In [ ]:
clip_of, win_of = windows["clip"].values, windows["win"].values
in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
assert not (in_train & in_val).any() and (in_train | in_val).all()

Y = np.zeros((len(windows), len(species)), np.float32)      # train: primary 1.0, secondary 0.5
for row, c in enumerate(clip_of):
    Y[row, sec_idx[c]] = CFG["secondary_weight"]
    Y[row, y_idx[c]] = 1.0
clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
W_row = (1.0 / clip_windows).astype(np.float32)             # each clip counts once

mean, std = E[in_train].mean(axis=0), E[in_train].std(axis=0) + 1e-6
Z = (E - mean) / std

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

# Validation rows, M1 protocol: first window, primary label only
rows_va = np.where(in_val & (win_of == 0))[0]
Y_va = np.zeros((len(rows_va), len(species)), np.float32)
Y_va[np.arange(len(rows_va)), y_idx[clip_of[rows_va]]] = 1.0

rows_all_va = np.where(in_val)[0]
va_groups = clip_of[rows_all_va]
order_clips = np.unique(va_groups)
Y_clip = np.zeros((len(order_clips), len(species)), np.float32)
Y_clip[np.arange(len(order_clips)), y_idx[order_clips]] = 1.0

FEATURES = {True: torch.from_numpy(Z), False: torch.from_numpy(E)}
Y_t = torch.from_numpy(Y); W_t = torch.from_numpy(W_row)
tr_rows = torch.from_numpy(np.where(in_train)[0])
print(f"Windows: train {in_train.sum()} | val {in_val.sum()} | val first-5s {len(rows_va)}", flush=True)

# Log prior of each class: same training rows, labels (secondary 0.5) and window weights as the loss
w_tr = W_row[in_train].astype(np.float64)
prior = (Y[in_train].astype(np.float64) * w_tr[:, None]).sum(axis=0) / w_tr.sum()
n_clipped = int(((prior < PRIOR_EPS) | (prior > 1 - PRIOR_EPS)).sum())
prior_clipped = np.clip(prior, PRIOR_EPS, 1 - PRIOR_EPS)
PRIOR_BIAS = torch.from_numpy(np.log(prior_clipped / (1 - prior_clipped)).astype(np.float32))
assert torch.isfinite(PRIOR_BIAS).all() and len(PRIOR_BIAS) == len(species)
print(f"Log-prior bias: mean {PRIOR_BIAS.mean():.4f} | min {PRIOR_BIAS.min():.4f} | max {PRIOR_BIAS.max():.4f} | "
      f"classes clipped {n_clipped}", flush=True)

## Heads and training loop

In [ ]:
def make_head(kind, dropout):
    if kind == "linear":
        return nn.Sequential(nn.Linear(EMB_DIM, len(species)))
    return nn.Sequential(nn.Linear(EMB_DIM, 512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512, len(species)))

@torch.no_grad()
def predict(model, X, rows):
    model.eval(); out = []
    for i in range(0, len(rows), 4096):
        out.append(torch.sigmoid(model(X[torch.from_numpy(rows[i:i + 4096])])).numpy())
    return np.concatenate(out)

def clip_max_auc(model, X):
    p = pd.DataFrame(predict(model, X, rows_all_va)).groupby(va_groups).max().loc[order_clips].values
    return macro_auc(Y_clip, p)[0]

def train_head(c):
    """Same loop as the A3 notebook. Clip-max is computed only when first-5s improves (same reported value, less CPU)."""
    torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
    X, epochs = FEATURES[c["standardised"]], c["epochs"]
    model = make_head(c["kind"], c["dropout"])
    if c["prior_bias"]:
        with torch.no_grad():
            model[-1].bias.copy_(PRIOR_BIAS)     # no random draw, so the weights and batches match the control's
    init_bias_mean = model[-1].bias.mean().item()
    print(f"{c['name']} initial bias mean {init_bias_mean:.4f}", flush=True)
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=c["max_lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=c["max_lr"], total_steps=epochs * steps, pct_start=0.1)
    best = dict(first=-1.0)
    first_epoch_loss = None
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows))]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            bce = nn.functional.binary_cross_entropy_with_logits(model(X[b]), Y_t[b], reduction="none")
            loss = (bce.mean(1) * W_t[b]).sum() / W_t[b].sum()
            opt.zero_grad(set_to_none=True)
            loss.backward(); opt.step(); sched.step()
            total += loss.item()
        if ep == 1:
            first_epoch_loss = total / steps
            print(f"{c['name']} first-epoch loss {first_epoch_loss:.4f}", flush=True)
        first, n_cls = macro_auc(Y_va, predict(model, X, rows_va))
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max_auc(model, X), best_epoch=ep, n_classes=n_cls); flag = "  <- best"
        if flag or ep % 5 == 0 or ep == epochs:
            print(f"{c['name']} ep {ep:2d}/{epochs} | loss {total / steps:.4f} | first-5s {first:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
    return {**best, "init_bias_mean": init_bias_mean, "first_epoch_loss": first_epoch_loss}

def cfg(kind, max_lr, dropout=None, standardised=True, prior_bias=False):
    tag = (f"{kind}_lr{max_lr:g}" + (f"_drop{dropout:g}" if dropout is not None else "") + ("" if standardised else "_raw")
           + ("_priorbias" if prior_bias else ""))
    return dict(name=tag, kind=kind, max_lr=max_lr, dropout=dropout, standardised=standardised, prior_bias=prior_bias,
                epochs=EPOCHS[kind])

# In priority order: if the time budget runs out, the last ones are skipped
SWEEP = [cfg("linear", 1e-3),                       # control: default bias, and the BASE row below
         cfg("linear", 1e-3, prior_bias=True),
         cfg("linear", 3e-3, prior_bias=True)]
BASE = SWEEP[0]["name"]

## Sweep

In [ ]:
def write_results(rows):
    done = [r for r in rows if r["status"] == "done"]
    base = next((r for r in done if r["name"] == BASE), None)
    payload = dict(device=DEVICE, torch_threads=torch.get_num_threads(), cfg=CFG, epochs=EPOCHS, budget_s=BUDGET_S,
                   baseline_reference=dict(first5s=BASELINE_FIRST, clip_max=BASELINE_CLIP_MAX),
                   baseline_diff_first5s=None if base is None else base["first"] - BASELINE_FIRST,
                   baseline_diff_clip_max=None if base is None else base["clip_max"] - BASELINE_CLIP_MAX,
                   prior_eps=PRIOR_EPS, prior_bias_mean=PRIOR_BIAS.mean().item(), prior_classes_clipped=n_clipped,
                   windows=len(E), notebook_seconds=time.time() - T0,
                   results=sorted(rows, key=lambda r: -r.get("first", -1.0)))
    tmp = OUT / "probe_retry_results.tmp.json"
    with open(tmp, "w") as f:
        json.dump(payload, f, indent=1)
    os.replace(tmp, OUT / "probe_retry_results.json")

rows, sec_per_epoch = [], {}
for c in SWEEP:
    # MLP cost before its first run: 4x linear, from a CPU timing of both heads at this batch size
    per_ep = sec_per_epoch.get(c["kind"], sec_per_epoch.get("linear", 0.0) * (4 if c["kind"] == "mlp" else 1))
    elapsed = time.time() - T0
    if elapsed + per_ep * c["epochs"] > BUDGET_S:
        print(f"SKIP {c['name']}: {elapsed:.0f}s elapsed + {per_ep * c['epochs']:.0f}s projected > budget {BUDGET_S:.0f}s", flush=True)
        rows.append({**c, "status": "skipped_time_budget"})
    else:
        t0 = time.time()
        best = train_head(c)
        sec = time.time() - t0
        sec_per_epoch[c["kind"]] = sec / c["epochs"]
        rows.append({**c, **best, "status": "done", "seconds": sec})
        print(f"TIME {c['name']}: {sec:.0f}s ({sec / c['epochs']:.1f}s/epoch) | first-5s {best['first']:.4f} | clip-max {best['clip_max']:.4f} | "
              f"initial bias mean {best['init_bias_mean']:.4f} | first-epoch loss {best['first_epoch_loss']:.4f}", flush=True)
        if c["name"] == BASE:
            d = best["first"] - BASELINE_FIRST
            print(f"BASELINE first-5s {best['first']:.4f} vs original {BASELINE_FIRST} (diff {d:+.4f}) | "
                  f"clip-max {best['clip_max']:.4f} vs {BASELINE_CLIP_MAX} (diff {best['clip_max'] - BASELINE_CLIP_MAX:+.4f})", flush=True)
            if abs(d) > 0.005:
                print("WARNING: baseline differs from the original by more than 0.005; compare configs against this run's baseline only", flush=True)
    write_results(rows)

print(f"\n{'config':28s} | {'first-5s':>8s} | {'clip-max':>8s} | {'best ep':>7s} | {'epochs':>6s} | {'sec':>6s} | {'init bias':>9s} | {'ep1 loss':>8s} | status")
for r in sorted(rows, key=lambda r: -r.get("first", -1.0)):
    if r["status"] == "done":
        print(f"{r['name']:28s} | {r['first']:8.4f} | {r['clip_max']:8.4f} | {r['best_epoch']:7d} | {r['epochs']:6d} | {r['seconds']:6.0f} | {r['init_bias_mean']:9.4f} | {r['first_epoch_loss']:8.4f} | done")
    else:
        print(f"{r['name']:28s} | {'-':>8s} | {'-':>8s} | {'-':>7s} | {r['epochs']:6d} | {'-':>6s} | {'-':>9s} | {'-':>8s} | {r['status']}")
print(f"\nNotebook total {time.time() - T0:.0f}s | wrote {OUT / 'probe_retry_results.json'}")